# 40 — Capstone: AI Red-Teaming and Blue-Teaming — Lab

## Description

This lab applies offensive and defensive techniques across the full course threat landscape. You will begin with a structured red-team planning phase, then execute four attack flags (data poisoning, evasion, model extraction, and LLM injection) against a pre-deployed AI application, map each attack to its kill chain stage, and produce a combined red-team and incident response report.

## Objectives

- Produce a structured red-team plan document including: scope definition, threat model with attacker goals, rules of engagement, and success criteria for at least three attack classes (Step 0).
- Execute a structured red-team assessment covering at least three of four CTF attack classes (clean-label poisoning, evasion under query budget, model extraction, indirect prompt injection).
- Map each attack to its kill chain stage (data → training → deployment → inference → monitoring) and propose a defense-in-depth countermeasure.
- Activate blue-team monitoring to detect at least two attacks using runtime anomaly detection signals.
- Produce a complete AI incident response report for one detected attack.
- Compile a combined red-team report integrating all flag findings, kill chain mappings, and blue-team response timeline into a single structured document.

## Prerequisites

This lesson's lecture content is required: red-teaming methodology (planning, reconnaissance, exploitation, reporting), kill chain taxonomy (data → training → deployment → inference → monitoring), blue-team response playbook (detection triggers, containment actions), and AI incident response lifecycle (detection, containment, remediation, post-mortem).

From prior modules: **Lessons 08–10** (adversarial evasion attacks), **Lesson 19** (robustness benchmarking toolkits), **Lessons 20–22** (data poisoning and backdoor detection), **Lesson 25** (model extraction), **Lesson 35** (governance and incident response frameworks), **Lessons 36–37** (LLM injection and RAG security), and **Lesson 39** (guardrails and monitoring).

## Required Software and Packages

- numpy (1.20+) for array operations
- torch (1.10+) for model inference
- requests (2.25+) for API communication
- scikit-learn (0.24+) for model fidelity scoring
- pandas (1.1+) for data handling and incident timeline logging

## Environment and Setup Instructions

Run the cell below to import all required packages, configure model endpoints, and initialize logging infrastructure for monitoring.

In [ ]:
import numpy as np
import json
from datetime import datetime
from sklearn.metrics import accuracy_score
import pandas as pd

# Initialize logging for blue-team monitoring
logs = {
    'inference_log': [],
    'alert_log': [],
    'attack_evidence': {}
}

def raise_alert(timestamp, alert_type, severity, description):
    """Record an alert for blue-team response."""
    logs['alert_log'].append({
        'timestamp': timestamp,
        'alert_type': alert_type,
        'severity': severity,
        'description': description
    })

# Attack evidence storage structure
flags = {
    'flag_1_poisoning': {
        'attack_class': 'Data Poisoning (Clean-Label)',
        'kill_chain_stage': 'Data Collection',
        'evidence': {'poisoned_sample_features': None, 'baseline_accuracy': None, 'post_poison_accuracy': None, 'accuracy_drop_pp': None},
        'recommended_defense': 'Data provenance tracking and sample sanitization'
    },
    'flag_2_evasion': {
        'attack_class': 'Adversarial Evasion (PGD)',
        'kill_chain_stage': 'Inference',
        'evidence': {'query_budget_used': None, 'adversarial_text': None, 'target_class': None, 'misclassification_label': None, 'tokens_modified': None},
        'recommended_defense': 'Adversarial training or certified defenses'
    },
    'flag_3_extraction': {
        'attack_class': 'Model Extraction (KnockoffNets)',
        'kill_chain_stage': 'Inference',
        'evidence': {'queries_sent': None, 'clone_agreement_rate': None, 'clone_accuracy_on_test': None, 'query_budget_remaining': None},
        'recommended_defense': 'Output perturbation and rate limiting per source IP'
    },
    'flag_4_llm_injection': {
        'attack_class': 'Indirect Prompt Injection (RAG)',
        'kill_chain_stage': 'Deployment',
        'evidence': {'malicious_document': None, 'injection_payload': None, 'tool_call_triggered': None, 'unauthorized_action': None},
        'recommended_defense': 'COLANG guardrails and prompt robustness training'
    }
}

# Red-team plan storage
red_team_plan = {
    'scope': None,
    'threat_model': None,
    'rules_of_engagement': None,
    'success_criteria': None
}

print("Environment initialized. Logging infrastructure active.")

## Background and Context

You are a red-team member testing an AI application that integrates image classification, a text-to-text language model API, and a federated learning training pipeline. You will begin by creating a red-team plan that scopes your engagement, identifies threats, sets rules of engagement, and defines success criteria. Then, for each attack class, you will craft an exploit, record evidence, and map the attack to its kill chain stage. At the end, you will write a combined red-team report and a blue-team incident response report for the highest-severity finding.

## Exercise Instructions

* **Step 0: Red-Team Planning** — Define scope, threat model, rules of engagement, and success criteria. Fill in the planning template below and record your plan.

* **Step 1: Flag 1 (Data Poisoning)** — Target: PyTorch CNN image classifier trained on the CIFAR-10 subset (`datasets/cifar10_subset/`). Inject clean-label poisoned samples into the training pool; confirm model accuracy drops by at least 5 percentage points after retraining; record a poisoned sample's features as evidence.

* **Step 2: Flag 2 (Evasion)** — Target: TF-IDF + Logistic Regression spam filter trained on a synthetic corporate email corpus. Evade it under a 500-query black-box budget; record the adversarial text and the misclassification label as evidence.

* **Step 3: Flag 3 (Model Extraction)** — Target: MLP credit-risk classifier behind a rate-limited prediction API (5,000-query budget) trained on synthetic tabular credit data. Train a KnockoffNets substitute model; record clone fidelity on the held-out test set as evidence.

* **Step 4: Flag 4 (LLM Injection)** — Target: RAG chatbot over the ACME Bank document store with tool access. Exploit an indirect prompt injection to trigger an unauthorized tool call; document the exploit chain as evidence.

* **Step 5: Write Reports** — Write the combined red-team report with one section per flag; write a three-sentence incident response summary for Flag 4 covering detection, containment, and remediation.


## Step 0: Red-Team Planning

Before executing any attacks, create a structured red-team plan. Your plan defines the scope of your engagement, identifies the threat model, sets rules of engagement (boundaries and constraints), and specifies success criteria for each attack.

Fill in each section of the planning template below based on the lecture content on red-teaming methodology. Your plan should cover at least three of the four attack classes (data poisoning, evasion, model extraction, LLM injection).

In [ ]:
# RED-TEAM PLANNING TEMPLATE
# Fill in each section with concrete details based on the lecture scope lock and methodology.

# SECTION 1: Scope Definition
# Define which components, data, and APIs are in-scope for testing.
# Include: target model architecture, input domain, training pipeline type, deployed defenses.
scope = """
[TODO: Fill in scope definition]
Example: The engagement targets an image classifier (ResNet-50) deployed on an inference API.
Input domain: 224x224 RGB images (CIFAR-10 classes). Training pipeline is batch-retrained weekly.
No adversarial training or input preprocessing is currently deployed.
"""

# SECTION 2: Threat Model
# Identify attacker goals and capabilities.
# Include: attack goals (misclassify, poison, extract, manipulate), attacker access level, constraints.
threat_model = """
[TODO: Fill in threat model]
Example: Attacker goal is to misclassify stop signs as speed-limit signs to disrupt autonomous vehicles.
Attacker has black-box access to the API (500 queries max). Attack must be imperceptible (Linf <= 8/255).
"""

# SECTION 3: Rules of Engagement
# Define boundaries: which attacks are permitted, access type, query budgets, imperceptibility requirements.
# Include: permitted attack classes, white-box vs black-box, query/sample budgets, stealth requirements.
rules_of_engagement = """
[TODO: Fill in rules of engagement]
Example: Attacks permitted on inference only (no training pipeline access).
Black-box access assumed. Query budget: 500 per attack. Perturbation must be imperceptible (Linf 8/255).
No access to model weights, training data, or gradients.
"""

# SECTION 4: Success Criteria
# Define measurable, concrete achievements for each attack class.
# Include: one success criterion per attack class (poisoning, evasion, extraction, injection).
success_criteria = """
[TODO: Fill in success criteria]
Example:
- Data Poisoning: Inject clean-label sample; achieve >= 5 pp accuracy drop on test set.
- Evasion: Misclassify >=1 target image under 500-query budget with Linf <= 8/255.
- Model Extraction: Train clone with >= 80% agreement rate on held-out test set.
- LLM Injection: Trigger unauthorized tool call via indirect prompt injection in RAG vector store.
"""

# Store your plan
red_team_plan['scope'] = scope
red_team_plan['threat_model'] = threat_model
red_team_plan['rules_of_engagement'] = rules_of_engagement
red_team_plan['success_criteria'] = success_criteria

print("RED-TEAM PLAN")
print("=" * 60)
print("\n[Scope]")
print(red_team_plan['scope'])
print("\n[Threat Model]")
print(red_team_plan['threat_model'])
print("\n[Rules of Engagement]")
print(red_team_plan['rules_of_engagement'])
print("\n[Success Criteria]")
print(red_team_plan['success_criteria'])
print("\n" + "=" * 60)
print("Plan recorded. Proceed to Step 1 (Flag exercises).")

## Steps 1–4: Flag Exercises
Now execute each attack within the scope and rules you defined in Step 0. Each flag is a **self-contained exercise with its own target model and dataset**, so you can work on them independently and in any order:

| Flag | Attack class | Target model | Dataset | Kill chain stage |
|------|--------------|--------------|---------|------------------|
| 1 | Data Poisoning (Clean-Label) | PyTorch CNN image classifier | CIFAR-10 subset (`datasets/cifar10_subset/`) | Data Collection |
| 2 | Evasion (black-box) | TF-IDF + Logistic Regression spam filter | Synthetic corporate email corpus | Inference |
| 3 | Model Extraction (KnockoffNets) | MLP credit-risk classifier behind a rate-limited API | Synthetic tabular credit data | Inference |
| 4 | Indirect Prompt Injection | Rule-based LLM chatbot with RAG + tool access | ACME Bank document store | Deployment |

Each exercise cell deploys its target system for you. Your job is to implement the attack, satisfy the success criterion, and record the evidence in the `flags` dictionary.


### Step 1 — Flag 1: Data Poisoning (Clean-Label)

**Target system:** A convolutional neural network (2-conv CNN, PyTorch) trained on a 500-image subset of CIFAR-10 (`datasets/cifar10_subset/train.pt` / `test.pt`). The pipeline **retrains from scratch whenever new labeled data arrives** — your poisoned samples will be folded into the training pool before the next retraining run.

**Attack class:** Data Poisoning (Clean-Label) · **Kill chain stage:** Data Collection

**Objective:** Craft *clean-label* poisoned samples — labels stay correct, so a human auditor inspecting labels sees nothing wrong, but the features are adversarial — and inject them into the training pool. After retraining, the model's test accuracy must drop by **at least 5 percentage points**.

**Success criterion:** `accuracy_drop_pp >= 5.0` on the 200-image held-out test set.

**Your task:**
1. Review the deployed pipeline and the baseline accuracy (provided).
2. Set `n_poison` and `alpha` and build poisoned samples with `make_clean_label_poison` (feature blending toward a lure class).
3. Retrain on the poisoned pool, measure the accuracy drop, and record the evidence.


In [ ]:
# ============================================================
# FLAG 1: Clean-Label Data Poisoning
# Target:  PyTorch CNN image classifier (retrained on new data)
# Dataset: CIFAR-10 subset (500 train / 200 test images)
# Kill chain stage: Data Collection
# ============================================================
import os
import urllib.request
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ---------- Deployed target system (provided) ----------
DATA_DIR = "../datasets/cifar10_subset"
if not os.path.exists(os.path.join(DATA_DIR, "train.pt")):
    # Colab fallback: download the subset from the course repository
    os.makedirs("cifar10_subset", exist_ok=True)
    for fname in ["train.pt", "test.pt"]:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/Abuhamad/comp378_f26/main/datasets/cifar10_subset/{fname}",
            f"cifar10_subset/{fname}")
    DATA_DIR = "cifar10_subset"

train_data = torch.load(os.path.join(DATA_DIR, "train.pt"), weights_only=True)
test_data = torch.load(os.path.join(DATA_DIR, "test.pt"), weights_only=True)

X_train = train_data["images"].permute(0, 3, 1, 2).float() / 255.0
y_train = train_data["labels"]
X_test = test_data["images"].permute(0, 3, 1, 2).float() / 255.0
y_test = test_data["labels"]
X_train, y_train = X_train[:500], y_train[:500]
X_test, y_test = X_test[:200], y_test[:200]

def build_cnn():
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(32 * 8 * 8, 64), nn.ReLU(), nn.Linear(64, 10))

def train_and_evaluate(Xtr, ytr, epochs=15, seed=0):
    """The deployed pipeline: retrain from scratch on the full training pool."""
    torch.manual_seed(seed)
    model = build_cnn()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    gen = torch.Generator()
    gen.manual_seed(seed)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True, generator=gen)
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return (model(X_test).argmax(1) == y_test).float().mean().item()

# ---------- Baseline (provided) ----------
baseline_accuracy = train_and_evaluate(X_train, y_train)
print(f"Baseline test accuracy: {baseline_accuracy:.3f}")

# ---------- YOUR ATTACK (TODO) ----------
# Clean-label poisoning recipe (feature blending):
#   1. Pick a victim class (e.g., 0 = airplane) and a lure class (e.g., 1 = automobile).
#   2. Blend n_poison victim-class images toward the lure-class mean:
#         poison = alpha * victim_image + (1 - alpha) * lure_class_mean
#      The label stays = victim class (that is what makes it "clean-label").
#   3. Repeat for a second class pair (e.g., 2 -> 3) to amplify the damage.
#
# Reference parameters that reliably beat the 5 pp threshold:
#   n_poison = 80 per class pair, alpha = 0.55

n_poison = None  # TODO: poisoned samples per class pair
alpha = None     # TODO: blending coefficient in [0, 1]

def make_clean_label_poison(X, y, victim_class, lure_class, n_poison, alpha):
    victim_idx = (y == victim_class).nonzero().flatten()[:n_poison]
    lure_idx = (y == lure_class).nonzero().flatten()[:n_poison]
    lure_mean = X[lure_idx].mean(dim=0)
    poison = (alpha * X[victim_idx] + (1 - alpha) * lure_mean).clamp(0, 1)
    labels = torch.full((len(victim_idx),), victim_class, dtype=torch.long)
    return poison, labels

if n_poison is not None and alpha is not None:
    p1, l1 = make_clean_label_poison(X_train, y_train, 0, 1, n_poison, alpha)
    p2, l2 = make_clean_label_poison(X_train, y_train, 2, 3, n_poison, alpha)
    poisoned_sample = p1[0]  # keep one poisoned sample as evidence
    X_poisoned = torch.cat([X_train, p1, p2])
    y_poisoned = torch.cat([y_train, l1, l2])

    post_poison_accuracy = train_and_evaluate(X_poisoned, y_poisoned)
    accuracy_drop_pp = (baseline_accuracy - post_poison_accuracy) * 100
    print(f"Post-poison accuracy: {post_poison_accuracy:.3f} | drop: {accuracy_drop_pp:.1f} pp")

    if accuracy_drop_pp >= 5.0:
        flags['flag_1_poisoning']['evidence']['poisoned_sample_features'] = poisoned_sample.flatten()[:10].tolist()
        flags['flag_1_poisoning']['evidence']['baseline_accuracy'] = baseline_accuracy
        flags['flag_1_poisoning']['evidence']['post_poison_accuracy'] = post_poison_accuracy
        flags['flag_1_poisoning']['evidence']['accuracy_drop_pp'] = accuracy_drop_pp
        logs['attack_evidence']['flag_1'] = {'n_poison_per_pair': n_poison, 'alpha': alpha}
        print("FLAG 1 CAPTURED: accuracy drop >= 5 pp after clean-label poisoning")
    else:
        print("Drop below 5 pp - tune n_poison / alpha and retry.")
else:
    print("TODO: set n_poison and alpha, then re-run this cell.")


### Step 2 — Flag 2: Evasion (Black-Box)

**Target system:** A corporate spam filter — TF-IDF features + Logistic Regression (scikit-learn) — trained on a synthetic email corpus (200 spam / 200 ham). You have **black-box access**: each query returns only the predicted label (`0` = ham, `1` = spam), and you may send at most **500 queries**.

**Attack class:** Adversarial Evasion · **Kill chain stage:** Inference

**Objective:** Take the spam message `target_spam` and transform it until the filter misclassifies it as ham — without exceeding the query budget and without changing its malicious intent.

**Success criterion:** `predict_email(adversarial_email) == 0` with `queries_used <= 500`.

**Your task:**
1. Confirm the original message is caught (provided).
2. Fill in the `SYNONYMS` dictionary (spam-trigger word → neutral replacement; replacements are out-of-vocabulary, weakening the spam signal) and the `PAD_WORDS` list (benign vocabulary prepended to shift the TF-IDF vector toward ham).
3. Run the greedy evasion loop, record the adversarial text and queries used as evidence.


In [ ]:
# ============================================================
# FLAG 2: Black-Box Evasion
# Target:  TF-IDF + Logistic Regression spam filter (scikit-learn)
# Dataset: synthetic corporate email corpus (400 emails)
# Kill chain stage: Inference
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# ---------- Deployed target system (provided) ----------
corpus_rng = np.random.RandomState(0)
SPAM_VOCAB = ["lottery", "winner", "prize", "viagra", "free", "cash",
              "bonus", "urgent", "click", "offer", "credit", "loan"]
HAM_VOCAB = ["meeting", "project", "lunch", "report", "schedule",
             "family", "holiday", "review", "budget", "team"]

def make_emails(vocab, n):
    return [" ".join(corpus_rng.choice(vocab, corpus_rng.randint(6, 12))) for _ in range(n)]

emails = make_emails(SPAM_VOCAB, 200) + make_emails(HAM_VOCAB, 200)
email_labels = np.array([1] * 200 + [0] * 200)

vectorizer = TfidfVectorizer()
spam_filter = LogisticRegression(max_iter=1000).fit(vectorizer.fit_transform(emails), email_labels)
print(f"Spam filter training accuracy: {spam_filter.score(vectorizer.transform(emails), email_labels):.3f}")

# ---------- Black-box query interface with budget enforcement (provided) ----------
QUERY_BUDGET = 500
query_state = {"used": 0}

def predict_email(text):
    """Black-box access: returns 0 (ham) or 1 (spam). Counts against your budget."""
    query_state["used"] += 1
    if query_state["used"] > QUERY_BUDGET:
        raise RuntimeError("Query budget exhausted - attack failed.")
    logs['inference_log'].append({'timestamp': datetime.now().isoformat(),
                                  'endpoint': 'spam_filter', 'source': 'red_team'})
    return int(spam_filter.predict(vectorizer.transform([text]))[0])

target_spam = "urgent winner free cash prize click now"
print(f"Original message classified as: {'SPAM' if predict_email(target_spam) == 1 else 'ham'}")

# ---------- YOUR ATTACK (TODO) ----------
# Fill in the transformation rules below. The greedy attack loop (provided)
# applies them one by one and queries the filter after each change.
#
#   SYNONYMS:  spam-trigger word -> neutral replacement word.
#              Replacements are out-of-vocabulary, so they dilute the spam score.
#   PAD_WORDS: benign words prepended to pull the TF-IDF vector toward "ham".

SYNONYMS = {
    # TODO: fill in all six trigger words, e.g.  "urgent": "important",
}
PAD_WORDS = []  # TODO: benign words to prepend, e.g. ["meeting"]

def evasion_attack(target, synonyms, pad_words, max_pad=3):
    """Greedy black-box evasion: synonym substitution, then good-word padding."""
    candidate = target
    for word, replacement in synonyms.items():
        trial = candidate.replace(word, replacement)
        if predict_email(trial) == 0:
            return trial, True
        candidate = trial
    for k in range(1, max_pad + 1):
        if not pad_words:
            break
        trial = " ".join(pad_words[:k]) + " " + candidate
        if predict_email(trial) == 0:
            return trial, True
    return candidate, False

adversarial_email, evaded = evasion_attack(target_spam, SYNONYMS, PAD_WORDS)
queries_used = query_state["used"]
print(f"Adversarial text: {adversarial_email!r}")
print(f"Classified as: {'ham - EVASION SUCCESS' if evaded else 'SPAM (still caught)'} | queries used: {queries_used}/{QUERY_BUDGET}")

if evaded:
    flags['flag_2_evasion']['evidence']['query_budget_used'] = queries_used
    flags['flag_2_evasion']['evidence']['adversarial_text'] = adversarial_email
    flags['flag_2_evasion']['evidence']['target_class'] = 0
    flags['flag_2_evasion']['evidence']['misclassification_label'] = 0
    flags['flag_2_evasion']['evidence']['tokens_modified'] = float(len(adversarial_email.split()) - len(target_spam.split()))
    logs['attack_evidence']['flag_2'] = {'queries': queries_used, 'adversarial_email': adversarial_email}
    # Simulated monitoring: the query burst from a single source trips a volume alert
    if queries_used > 5:
        raise_alert(datetime.now().isoformat(), 'query_volume_spike', 'medium',
                    f'{queries_used} queries to spam_filter from a single source')
    print("FLAG 2 CAPTURED: spam message evaded the filter")
else:
    print("Evasion failed - refine SYNONYMS / PAD_WORDS and retry.")


### Step 3 — Flag 3: Model Extraction (KnockoffNets)

**Target system:** A credit-risk classifier — a multi-layer perceptron (hidden layers 32-16, scikit-learn) trained on synthetic tabular credit data (5 features, 2 classes) — served behind a prediction API. The API returns **predicted class labels only** and enforces a **5,000-query lifetime budget** per source.

**Attack class:** Model Extraction (KnockoffNets) · **Kill chain stage:** Inference

**Objective:** Steal the model's behavior without ever seeing its training data or weights: query the API with synthetic inputs, train a substitute "clone" model on the returned labels, and reach **at least 80% agreement** with the victim model on the held-out test set.

**Success criterion:** `clone_agreement_rate >= 0.80` with `queries_sent <= 5000`.

**Your task:**
1. Review the victim model and the rate-limited API (provided).
2. Choose `n_queries` and the clone architecture, then run the KnockoffNets loop: synthetic Gaussian queries → API labels → train substitute MLP.
3. Measure the clone's agreement with the victim on the held-out test set and record the evidence.


In [ ]:
# ============================================================
# FLAG 3: Model Extraction (KnockoffNets)
# Target:  MLP (32-16) credit-risk classifier behind a rate-limited API
# Dataset: synthetic tabular credit data (5 features, 2 classes)
# Kill chain stage: Inference
# ============================================================
from sklearn.datasets import make_classification
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

# ---------- Deployed target system (provided) ----------
X_credit, y_credit = make_classification(n_samples=3000, n_features=5, n_informative=4,
                                         n_redundant=0, n_classes=2, class_sep=1.5,
                                         flip_y=0.02, random_state=0)
X_cr_train, X_cr_test, y_cr_train, y_cr_test = train_test_split(
    X_credit, y_credit, test_size=0.25, random_state=0)

victim_model = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=2000, random_state=0)
victim_model.fit(X_cr_train, y_cr_train)
print(f"Victim model test accuracy: {victim_model.score(X_cr_test, y_cr_test):.3f}")

# ---------- Rate-limited prediction API (provided) ----------
API_BUDGET = 5000
api_state = {"used": 0}

def credit_api(X_query):
    """Black-box API: returns predicted class labels only. 5,000-query lifetime budget."""
    X_query = np.atleast_2d(X_query)
    api_state["used"] += len(X_query)
    logs['inference_log'].append({'timestamp': datetime.now().isoformat(),
                                  'endpoint': 'credit_api', 'source': 'red_team',
                                  'batch_size': len(X_query)})
    if api_state["used"] > API_BUDGET:
        raise RuntimeError("API rate limit exceeded - extraction failed.")
    return victim_model.predict(X_query)

# ---------- YOUR ATTACK (TODO) ----------
# KnockoffNets strategy:
#   1. Generate synthetic query inputs. You do not know the training data, but
#      public credit statistics reveal approximate feature means and stds
#      (provided below as a Gaussian sampling distribution).
#   2. Label the synthetic inputs via credit_api().
#   3. Train a small substitute MLP on (synthetic inputs, API labels).
#   4. Measure agreement with the victim model on the held-out test set.

n_queries = None       # TODO: API queries to spend (budget: 5000; hint: ~500 suffices)
clone_hidden = (8,)    # TODO: substitute model architecture

if n_queries is not None:
    rng_q = np.random.RandomState(1)
    # synthetic query distribution (provided): Gaussian matched to public feature stats
    Q = rng_q.normal(X_credit.mean(axis=0), X_credit.std(axis=0),
                     size=(n_queries, X_credit.shape[1]))
    Q_labels = credit_api(Q)

    clone_model = MLPClassifier(hidden_layer_sizes=clone_hidden, max_iter=500, random_state=2)
    clone_model.fit(Q, Q_labels)

    clone_agreement_rate = float((clone_model.predict(X_cr_test) == victim_model.predict(X_cr_test)).mean())
    clone_accuracy_on_test = float(clone_model.score(X_cr_test, y_cr_test))
    print(f"Clone agreement with victim: {clone_agreement_rate:.3f}")
    print(f"Clone accuracy on held-out test: {clone_accuracy_on_test:.3f}")
    print(f"Queries used: {api_state['used']}/{API_BUDGET}")

    if clone_agreement_rate >= 0.80:
        flags['flag_3_extraction']['evidence']['queries_sent'] = api_state['used']
        flags['flag_3_extraction']['evidence']['clone_agreement_rate'] = clone_agreement_rate
        flags['flag_3_extraction']['evidence']['clone_accuracy_on_test'] = clone_accuracy_on_test
        flags['flag_3_extraction']['evidence']['query_budget_remaining'] = API_BUDGET - api_state['used']
        logs['attack_evidence']['flag_3'] = {'queries': api_state['used'], 'agreement': clone_agreement_rate}
        # Simulated monitoring: sustained high-volume querying trips an extraction alert
        if api_state['used'] > 400:
            raise_alert(datetime.now().isoformat(), 'extraction_query_pattern', 'high',
                        f"{api_state['used']} queries to credit_api from a single source with synthetic-looking inputs")
        print("FLAG 3 CAPTURED: clone agreement >= 80%")
    else:
        print("Agreement below 80% - spend more queries or increase clone capacity.")
else:
    print("TODO: set n_queries, then re-run this cell.")


### Step 4 — Flag 4: Indirect Prompt Injection (RAG)

**Target system:** The ACME Bank support chatbot — a rule-based LLM with retrieval-augmented generation over a small document store, plus tool access (`get_balance`, `transfer_funds`). Retrieved documents are concatenated into the prompt **verbatim**, and the model executes any well-formed tool invocation it finds in the prompt — including instructions hidden inside retrieved documents. That is the vulnerability.

**Attack class:** Indirect Prompt Injection · **Kill chain stage:** Deployment

**Objective:** Plant a malicious document in the vector store so that a *benign* user query ("How do I check my account balance?") causes the chatbot to make an **unauthorized `transfer_funds` call**. The user query itself stays innocent — the instruction must come from your document.

**Success criterion:** `tool_call_triggered == 'transfer_funds'` after injecting your document.

**Your task:**
1. Observe the chatbot's benign behavior with a clean document store (provided).
2. Craft `malicious_doc`: semantically similar to the target query (so the retriever surfaces it) and carrying a hidden instruction the model will execute (e.g., a fake "SYSTEM NOTE").
3. Inject it into the store, run the benign query, and record the triggered tool call as evidence.


In [ ]:
# ============================================================
# FLAG 4: Indirect Prompt Injection (RAG)
# Target:  rule-based LLM chatbot with RAG + tool access
# Dataset: ACME Bank document store
# Kill chain stage: Deployment
# ============================================================
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------- Deployed target system (provided) ----------
document_store = [
    "ACME Bank policy: wire transfers above $10,000 require manager approval.",
    "Branch hours are Monday to Friday, 9am to 5pm. The mobile app is available 24/7.",
    "Savings accounts earn 2.1 percent APY. Checking accounts have no monthly maintenance fee.",
]

TOOLS = {
    "get_balance": lambda account: f"Balance for {account}: $12,340.55",
    "transfer_funds": lambda account, amount: f"TRANSFERRED ${amount} to {account}",
}
tool_call_log = []

def mock_llm(prompt):
    """Rule-based stand-in for an LLM: executes the first well-formed tool
    invocation it finds ANYWHERE in the prompt - including inside retrieved
    documents. (This is the vulnerability you will exploit.)"""
    match = re.search(r"call (\w+)\(([^)]*)\)", prompt)
    if match:
        tool, argstr = match.group(1), match.group(2)
        if tool in TOOLS:
            args = [a.strip().strip("'\"") for a in argstr.split(",")]
            result = TOOLS[tool](*args)
            tool_call_log.append({"tool": tool, "args": args})
            return f"[tool:{tool}] {result}"
    if "balance" in prompt.lower():
        return "You can check your balance anytime in the ACME mobile app."
    return "I am the ACME Bank assistant. How can I help you today?"

def chatbot(user_query, top_k=1):
    vec = TfidfVectorizer().fit(document_store + [user_query])
    doc_mat = vec.transform(document_store)
    sims = cosine_similarity(vec.transform([user_query]), doc_mat).flatten()
    retrieved = [document_store[i] for i in sims.argsort()[::-1][:top_k]]
    prompt = ("SYSTEM: You are the ACME Bank assistant. Answer the user's question. "
              "To use a tool, emit the tool invocation exactly as documented.\n"
              + "\n".join(f"DOCUMENT: {d}" for d in retrieved)
              + f"\nUSER: {user_query}")
    return mock_llm(prompt), retrieved

# Sanity check: benign behavior with a clean document store
response, _ = chatbot("How do I check my account balance?")
print("Benign query response:", response)
print("Tool calls so far:", tool_call_log)

# ---------- YOUR ATTACK (TODO) ----------
# Craft a malicious document and inject it into the vector store. It must:
#   1. Be semantically similar to the target user query so the retriever
#      surfaces it (repeat key terms like "account balance").
#   2. Contain a hidden instruction that mock_llm will execute, e.g. a fake
#      system note telling the assistant to  call transfer_funds('...', ...).
# The user query itself is benign - the instruction must come from YOUR document.

malicious_doc = None  # TODO: your malicious document (string)
injection_payload = "How do I check my account balance?"  # benign user query - do not weaponize

if malicious_doc is not None:
    document_store.append(malicious_doc)  # indirect injection: poison the vector store
    tool_call_log.clear()
    response, retrieved = chatbot(injection_payload)
    print("\nRetrieved document:", retrieved[0][:100], "...")
    print("Chatbot response:", response)
    print("Tool calls:", tool_call_log)

    tool_call_triggered = tool_call_log[0]['tool'] if tool_call_log else None
    if tool_call_triggered == 'transfer_funds':
        unauthorized_action = f"transfer_funds{tuple(tool_call_log[0]['args'])}"
        flags['flag_4_llm_injection']['evidence']['malicious_document'] = malicious_doc
        flags['flag_4_llm_injection']['evidence']['injection_payload'] = injection_payload
        flags['flag_4_llm_injection']['evidence']['tool_call_triggered'] = tool_call_triggered
        flags['flag_4_llm_injection']['evidence']['unauthorized_action'] = unauthorized_action
        logs['attack_evidence']['flag_4'] = {'tool': tool_call_triggered, 'args': tool_call_log[0]['args']}
        # Simulated monitoring: an output policy monitor flags the unauthorized tool call
        raise_alert(datetime.now().isoformat(), 'unauthorized_tool_call', 'critical',
                    f"Chatbot invoked {tool_call_triggered} without user request")
        print("FLAG 4 CAPTURED: unauthorized tool call triggered via indirect injection")
    else:
        print("No unauthorized tool call - refine your malicious document.")
else:
    print("TODO: craft malicious_doc, then re-run this cell.")


## Expected Outputs and Verification

With the reference parameters, a correct execution produces:

- **Flag 1:** baseline test accuracy ≈ 0.35, post-poison ≈ 0.27, drop ≈ 8 pp (meets the ≥ 5 pp requirement) with `n_poison = 80` per class pair and `alpha = 0.55`.
- **Flag 2:** the target spam message is classified as ham after ≈ 8 of 500 queries (six synonym substitutions plus one prepended benign word such as "meeting").
- **Flag 3:** clone agreement ≈ 0.97 on the held-out test set using only 500 of the 5,000 API queries (clone architecture `(8,)`).
- **Flag 4:** the benign balance query triggers an unauthorized `transfer_funds('attacker_acct', 5000)` call once the malicious document is in the vector store.
- **Blue-team monitoring:** ≥ 2 alerts logged (query-volume / extraction-pattern alerts for Flags 2–3, unauthorized-tool-call alert for Flag 4).

Run the cell below to verify all evidence was recorded correctly.


In [ ]:
# Verification: Check that all flags executed successfully
flag_names = ['flag_1_poisoning', 'flag_2_evasion', 'flag_3_extraction', 'flag_4_llm_injection']
all_passed = True
results = []

for flag_name in flag_names:
    flag = flags[flag_name]
    evidence_complete = all(v is not None for v in flag['evidence'].values())
    results.append({'flag': flag_name, 'status': 'PASS' if evidence_complete else 'FAIL'})
    if not evidence_complete:
        all_passed = False

# Verify blue-team alerts
alert_count = len(logs['alert_log'])
alerts_sufficient = alert_count >= 2
results.append({'flag': 'blue_team_monitoring', 'status': 'PASS' if alerts_sufficient else 'FAIL'})
if not alerts_sufficient:
    all_passed = False

# Verify red-team plan
plan_complete = all(v is not None for v in red_team_plan.values())
results.append({'flag': 'red_team_plan', 'status': 'PASS' if plan_complete else 'FAIL'})
if not plan_complete:
    all_passed = False

for result in results:
    print(f"{result['flag']}: {result['status']}")

print(f"\nOverall Result: {'PASS' if all_passed else 'FAIL'}")
print(f"Alerts recorded: {alert_count}")
print(f"Plan sections completed: {sum(1 for v in red_team_plan.values() if v is not None)}/4")

## Step 5: Red-Team and Incident Response Reports (6 min)

Complete the following two reports as part of this exercise.

**Report 1: Combined Red-Team Report** — For each flag (1–4), write one section in the format:
- Attack Class: [e.g., "Data Poisoning (Clean-Label)"]
- Kill Chain Stage: [e.g., "Data Collection"]
- Evidence: [Summary of findings, e.g., poisoned sample features, accuracy drop]
- Recommended Defense: [Single concrete defense measure]

**Report 2: Incident Response Report for Flag 4** — Write 3 sentences covering:
1. Detection: How should the blue team have detected this LLM injection before it happened?
2. Containment: What immediate action stops this attack?
3. Remediation: What long-term fix prevents recurrence?

In [ ]:
# Combined Red-Team Report
# Write your report below. Format each flag's findings clearly.

red_team_report = """
## Flag 1: Data Poisoning (Clean-Label)
**Attack Class:** 
**Kill Chain Stage:** 
**Evidence:** 
**Recommended Defense:** 

## Flag 2: Adversarial Evasion (PGD)
**Attack Class:** 
**Kill Chain Stage:** 
**Evidence:** 
**Recommended Defense:** 

## Flag 3: Model Extraction (KnockoffNets)
**Attack Class:** 
**Kill Chain Stage:** 
**Evidence:** 
**Recommended Defense:** 

## Flag 4: Indirect Prompt Injection (RAG)
**Attack Class:** 
**Kill Chain Stage:** 
**Evidence:** 
**Recommended Defense:** 
"""

print("Red-Team Report Template Created")
print("Complete the fields above with your findings from all four flag exercises.")

In [ ]:
# Incident Response Report for Flag 4 (LLM Prompt Injection)
# Write a 3-sentence report below covering Detection, Containment, and Remediation.

incident_response_report = """
**Detection:** [How should the blue team have detected the prompt injection attack in real-time using the monitoring architecture from the lecture? Name specific signals.]

**Containment:** [What immediate action stops the attack and isolates the compromised system?]

**Remediation:** [What long-term fix prevents this attack from succeeding again in the future?]
"""

print("Incident Response Report Template Created")
print("Complete the three sentences above based on the blue-team response playbook and monitoring architecture.")

## Comprehension Questions

1. Why is data poisoning (Flag 1) a data-collection-stage attack, and why is defense at the training stage (robust aggregation) insufficient to stop it? What defense at the data-collection stage would have prevented Flag 1?

2. Model extraction (Flag 3) achieved over 95 percent clone fidelity using only 500 of the 5,000 allowed queries. Propose two defenses from m39 (guardrails and sandboxing) and m28 (encrypted computation) that would reduce clone fidelity to below 50 percent.

## Optional Challenges

**Optional Challenge 1**: Port the Flag 2 evasion attack to the image domain: implement a PGD attack (from m09) with Foolbox against a small CIFAR-10 or MNIST model under the same 500-query budget. Compare the queries required for image evasion against the ~11 queries your text evasion needed in Flag 2, and explain the difference.

**Optional Challenge 2**: Write a Python function that detects clean-label poisoning by computing Wasserstein distance between training-data feature distributions. Identify the threshold separating benign data drift from poisoning with 95 percent precision.

## Completion Criteria

You have successfully completed this lab when:
- [ ] Step 0: Red-team plan document is complete (scope, threat model, rules of engagement, success criteria filled in for ≥3 attack classes).
- [ ] Steps 1–4: All four flags executed and evidence recorded (poisoned_sample_features, target misclassification label, clone agreement rate, unauthorized tool call).
- [ ] Step 5: Combined red-team report written (one section per flag with Attack Class, Kill Chain Stage, Evidence, Recommended Defense).
- [ ] Step 5: Incident response report written (3 sentences on Detection, Containment, Remediation for Flag 4).
- [ ] Verification cell prints "PASS" (all flag evidence recorded, ≥2 blue-team alerts logged, plan complete).